In [1]:
import sys
sys.path.insert(1, 'CMAPSS')
import eval_utils
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import tensorflow as tf
from tensorflow import keras

from sklearn.preprocessing import MinMaxScaler

import matplotlib.pyplot as plt
from matplotlib import gridspec
import json
import os
import random
import warnings
from preProcessingData          import PreProcessData


os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # or any {'0', '1', '2'}
warnings.filterwarnings('ignore') 

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  


2024-10-16 17:48:11.034901: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-16 17:48:11.106323: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-10-16 17:48:11.125680: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-16 17:48:11.563398: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: li

In [7]:
with open('model/modelConfig.json', 'r') as configFile:
        config = json.load(configFile)
kanConfig = config["kan"]  

# ---------------------------------    
# Reactor, Separator,Stripper
# ---------------------------------
component  = kanConfig["component"]
modelName  = "kan"

os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
# strategy = tf.distribute.MirroredStrategy()

np.random.seed(1)
random.seed(1)
# ---------------------------------
# Pre processing data
# ---------------------------------
preData                         = PreProcessData()
rawData                         = preData.loadCriticalData()                                            # Load the data from critical file
# addedVariablesData              = preData.additionEfficiencyVar(rawData)                              # Add some additional efficiency variable
normalizedAllData               = preData.normalizeBuilinFunction(rawData)                              # Normalize the all data
# normalizedHealthy,healthyPos    = preData.getHealthyState(normalizedAllData)                            # Load the healthy state
effTrajectories                 = preData.getTrajectory(normalizedAllData, getValueOnly= True, headline= preData.beauEfficiencyCr1)         # Extract the trajectory
maniTrajectories                = preData.getTrajectory(normalizedAllData, getValueOnly= True, headline= preData.beauManipulatedCr1)         # Extract the trajectory

# ---------------------------------
# Create the cycles
# ---------------------------------
hours                           = preData.addFloatTime(rawData)
cycleTrajectories               = preData.getTrajectory(hours)         # Extract the trajectory

# print("----------------------------")
# print(effTrajectories)

trainEff, trainMani,trainCycle, \
valEffData, valManiData, valCycleData = preData.splitTrainValTraject(effTrajectories, maniTrajectories,cycleTrajectories)

print(len(trainEff), len(trainMani), len(trainCycle), len(valEffData), len(valManiData), len(valCycleData))
# ---------------------------
# All data
# --------------------------
trainSeqEffData,_     = preData.splitAllDataIntoSequences(trainEff)
trainSeqManiData,_    = preData.splitAllDataIntoSequences(trainMani)
trainSeqCycleData,_   = preData.splitAllDataIntoSequences(trainCycle)
trainSeqCycleData     = np.mean(trainSeqCycleData, axis=1)

valSeqEffData,_     = preData.splitAllDataIntoSequences(valEffData)
valSeqManiData,_    = preData.splitAllDataIntoSequences(valManiData)
valSeqCycleData,_   = preData.splitAllDataIntoSequences(valCycleData)
valSeqCycleData     = np.mean(valSeqCycleData, axis=1)

print(f"The size of the train data: {trainSeqEffData.shape} {trainSeqManiData.shape} {trainSeqCycleData.shape}")
print(f"The size of the test data: {valSeqEffData.shape} {valSeqManiData.shape} {valSeqCycleData.shape}")
print(type(trainSeqCycleData))

XDIM = trainSeqEffData.shape[2]
WDIM = trainSeqManiData.shape[2]
# ---------------------------
# Single sample
# ---------------------------
# singleTraject   = trajectory["Lubricación  deficiente"][0][preData.beauEfficiencyCr1 + preData.addEfficiencyCr1]
# seqData,_       = preData.splitAndPadSequencesTensor([singleTraject,singleTraject],seqLength=kanConfig['seqLength'], overlap= kanConfig['overlap'])

# trajectory.keys()

Process:63093||2024-10-16 18:18:09,368-INFO [preProcessingData.py:loadCriticalData():218]:Loading critical MOF data from files ...
Process:63093||2024-10-16 18:18:09,660-INFO [preProcessingData.py:loadCriticalData():224]:Finish loading data from files ...
Process:63093||2024-10-16 18:18:10,952-INFO [preProcessingData.py:splitAllDataIntoSequences():687]:Spliting data into sequences now ...


9 9 9 6 6 6


Process:63093||2024-10-16 18:18:14,000-INFO [preProcessingData.py:splitAllDataIntoSequences():687]:Spliting data into sequences now ...
Process:63093||2024-10-16 18:18:17,102-INFO [preProcessingData.py:splitAllDataIntoSequences():687]:Spliting data into sequences now ...
Process:63093||2024-10-16 18:18:21,698-INFO [preProcessingData.py:splitAllDataIntoSequences():687]:Spliting data into sequences now ...
Process:63093||2024-10-16 18:18:23,925-INFO [preProcessingData.py:splitAllDataIntoSequences():687]:Spliting data into sequences now ...
Process:63093||2024-10-16 18:18:25,999-INFO [preProcessingData.py:splitAllDataIntoSequences():687]:Spliting data into sequences now ...


The size of the train data: (41570, 512, 9) (41570, 512, 6) (41570,)
The size of the test data: (33864, 512, 9) (33864, 512, 6) (33864,)
<class 'numpy.ndarray'>


In [3]:
class C_constraint(tf.keras.layers.Layer):   # Inheritance class        
    def call(self, inputdata):
        x, t = inputdata
        print("--------------------------====================")
        print(x, t)

        x_mu = tf.reduce_mean(x)
        t_mu = tf.reduce_mean(t)
        cov = tf.reduce_sum((x-x_mu)*(t-t_mu))
        print("cod",cov)
        std = tf.math.pow(tf.reduce_sum((x-x_mu)**2),0.5) * tf.math.pow(tf.reduce_sum((t-t_mu)**2),0.5)
        loss = cov/std
        loss = tf.reduce_mean(loss)
        
        self.add_loss(1*loss)
        self.add_metric(loss, aggregation='mean', name='l1')
        return x

    
class NG_constraint(tf.keras.layers.Layer):   # Inheritance class
    def __init__(self,alpha,**kwargs):  
        super().__init__(**kwargs)
        self.alpha = alpha
        
    def call(self, inputdata):
        x,t = inputdata
        x_diff = x[1:]-x[:-1]
        t_diff = t[1:]-t[:-1]
        grad = x_diff/t_diff
        loss = tf.reduce_sum(tf.math.maximum(tf.constant(0.0),grad))
        self.add_loss(self.alpha*loss)
        self.add_metric(loss, aggregation='mean', name='Grad')

        return x
    
class F_constraint(tf.keras.layers.Layer):   # Inheritance class
    def __init__(self,alpha,**kwargs):  
        super().__init__(**kwargs)
        self.alpha = alpha
        
        self.A = 0.012
        self.B = 4.95
        self.C = 0.89
        self.BETA = 6.7
        self.P = 0.5
        
    def call(self, inputdata):
        x, t = inputdata
        HI = self.C - ((t * (-np.log(1 - self.P))**(-1/self.BETA))*self.A)**self.B
        mask = tf.math.less(HI,0)
        HI = tf.where(mask,0.0,HI)
        loss = tf.reduce_mean(tf.abs(x-HI))
        
        self.add_loss(self.alpha*loss)
        self.add_metric(loss, aggregation='mean', name='l1')
        return x
    
def H_index_Proposal_CNN(latent_dims,Window_size = 100,z_size=1,cnn_dim = 85,stride = 1,alpha = 1,constraint = 'C'):
    
    x_dim = XDIM
    w_dim = WDIM
    
    inpX    = tf.keras.layers.Input(shape=(Window_size,x_dim),name = 'InpX')
    inputT  = tf.keras.layers.Input(shape=(1,),name = 'inpT')
    inpW    = tf.keras.layers.Input(shape=(Window_size,w_dim),name = 'inpW')
    
    print("-------------------------------------")
    print(inpX.shape, inputT.shape, inpW.shape)
    

    x       = inpX
    w_in    = inpW
    
    for i in latent_dims:
        x=tf.keras.layers.Conv1D(i,cnn_dim,stride,activation='relu',padding = 'same')(x)

    x = tf.keras.layers.Flatten(name = 'flatten')(x)
    x = tf.keras.layers.Dense(z_size,name="Z")(x) #activity_regularizer=regularizers.l1(10e-5)
    
    if constraint == 'C':
        # x = C_constraint(alpha, name='constrained')([x, inputT])
        x = C_constraint()([x, inputT])

    elif constraint == 'NG':
        x = tf.keras.layers.Lambda(lambda x: tf.math.l2_normalize(x), name='Z_after')(x)
        x = NG_constraint(alpha, name='constrained')([x, inputT])
    elif constraint == 'F':
        x = F_constraint(alpha, name='constrained')([x, inputT])
    else:
        raise ValueError("Invalid constraint type. Must be 'C', 'NG', or 'F'") 
    
    x = tf.keras.layers.RepeatVector(Window_size,name='repeat')(x)
    
    x = tf.keras.layers.Concatenate(axis  =-1)([x,w_in])
    for i in reversed(latent_dims):
        x = tf.keras.layers.Conv1D(i,cnn_dim,strides=1,activation='relu',padding = 'same')(x)
    x=tf.keras.layers.Dense(x_dim,name='X')(x)   
       
    model = tf.keras.models.Model([inpX,inputT,inpW],x)
    return model

In [4]:
constraint  = 'C'
layers      =  [128,64,32,16]
Window_size = 512
model       = H_index_Proposal_CNN(layers,Window_size,constraint = constraint)

-------------------------------------
(None, 512, 9) (None, 512, 1) (None, 512, 6)
--------------------------====================
Tensor("Placeholder:0", shape=(None, 1), dtype=float32) Tensor("Placeholder_1:0", shape=(None, 512, 1), dtype=float32)
cod Tensor("c_constraint/Sum:0", shape=(), dtype=float32)


2024-10-16 17:48:30.114466: E tensorflow/stream_executor/cuda/cuda_driver.cc:265] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2024-10-16 17:48:30.114497: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:169] retrieving CUDA diagnostic information for host: enit23-005
2024-10-16 17:48:30.114505: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:176] hostname: enit23-005
2024-10-16 17:48:30.114604: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:200] libcuda reported version is: 555.58.2
2024-10-16 17:48:30.114627: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:204] kernel reported version is: 555.58.2
2024-10-16 17:48:30.114634: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:310] kernel version seems to match DSO: 555.58.2
2024-10-16 17:48:30.115092: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions

In [5]:
OPTIMIZER   = tf.keras.optimizers.Adam(learning_rate=0.001)
LOSS        = tf.keras.losses.MeanAbsoluteError()
model.compile(optimizer=OPTIMIZER, loss=LOSS, metrics=['mae'])
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

train_ds = tf.data.Dataset.from_tensor_slices(((trainSeqEffData, trainSeqCycleData, trainSeqManiData), trainSeqEffData))
train_ds = train_ds.batch(32).shuffle(buffer_size=trainSeqEffData.shape[0])

history = model.fit(train_ds, epochs=50, verbose=1)

plt.plot(history.history["loss"])
# plt.plot(history.history["val_loss"])
plt.title("model loss")
plt.ylabel("loss")
plt.xlabel("epoch")
plt.legend(["train"],loc ="upper left")

Num GPUs Available:  0
Epoch 1/50
--------------------------====================
Tensor("model/Z/BiasAdd:0", shape=(None, 1), dtype=float32) Tensor("model/Cast_1:0", shape=(None, 512, 1), dtype=float32)
cod Tensor("model/c_constraint/Sum:0", shape=(), dtype=float32)
--------------------------====================
Tensor("model/Z/BiasAdd:0", shape=(None, 1), dtype=float32) Tensor("model/Cast_1:0", shape=(None, 512, 1), dtype=float32)
cod Tensor("model/c_constraint/Sum:0", shape=(), dtype=float32)


InvalidArgumentError: Graph execution error:

Detected at node 'gradient_tape/model/c_constraint/mul/BroadcastGradientArgs' defined at (most recent call last):
    File "/home/duc/anaconda3/envs/tensorflow2.10.0Crusher/lib/python3.10/runpy.py", line 196, in _run_module_as_main
      return _run_code(code, main_globals, None,
    File "/home/duc/anaconda3/envs/tensorflow2.10.0Crusher/lib/python3.10/runpy.py", line 86, in _run_code
      exec(code, run_globals)
    File "/home/duc/anaconda3/envs/tensorflow2.10.0Crusher/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
      app.launch_new_instance()
    File "/home/duc/anaconda3/envs/tensorflow2.10.0Crusher/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
      app.start()
    File "/home/duc/anaconda3/envs/tensorflow2.10.0Crusher/lib/python3.10/site-packages/ipykernel/kernelapp.py", line 739, in start
      self.io_loop.start()
    File "/home/duc/anaconda3/envs/tensorf

In [ ]:
# print("Saving the model ...............")
# model.save('ckpt/VAEConstrains/my_model.h5')  # Save as an HDF5 file

model       = keras.models.load_model('ckpt/VAEConstrains/my_model.h5', custom_objects={'C_constraint': C_constraint})
# Show the model architecture
model.summary()
print("load model successfully")

In [ ]:
laten_extract = tf.keras.models.Model(inputs=[model.inputs[0], model.inputs[1], model.inputs[2]],
                                               outputs=[model.get_layer("Z").output])

z       = laten_extract.predict([trainSeqEffData,trainSeqManiData,trainSeqCycleData], batch_size=32)
scaler  = MinMaxScaler()
z       = scaler.fit_transform(z.reshape(-1,1))
plt.plot(z[0:200], label = 'Predicted HI')
